# Model Training, Evaluation & MLflow Integration

This notebook trains Prophet and LSTM models, evaluates performance, and logs experiments to MLflow.

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.insert(0, '..')

import mlflow
import mlflow.sklearn
from datetime import datetime, timedelta

from config import settings
from logger import logger
from ingestion.cbk_stats import CBKMPesaClient
from models.prophet_model import ProphetForecaster
from models.lstm_model import LSTMForecaster
from models.ensemble import EnsembleForecaster
from models.evaluate import ModelEvaluator
from features.feature_engineering import FeatureEngineering

# MLflow setup
mlflow.set_tracking_uri(settings.MLFLOW_TRACKING_URI)
mlflow.set_experiment(settings.MLFLOW_EXPERIMENT_NAME)

print("✓ All libraries and configurations loaded")

## 2. Load and Prepare Data

In [ ]:
# Ingest data
cbk_client = CBKMPesaClient()
mpesa_data = cbk_client.get_mpesa_statistics()

# Create time series
series = pd.Series(
    mpesa_data['total_transactions'].values,
    index=pd.to_datetime(mpesa_data['date'])
)

# Train/test split (80/20)
split_point = int(len(series) * 0.8)
train_series = series[:split_point]
test_series = series[split_point:]

print(f"📊 Data loaded successfully")
print(f"   Training set: {len(train_series)} records ({train_series.index[0].date()} to {train_series.index[-1].date()})")
print(f"   Test set: {len(test_series)} records ({test_series.index[0].date()} to {test_series.index[-1].date()})")

## 3. Train Prophet Model

In [ ]:
# Initialize and train Prophet
prophet_model = ProphetForecaster(
    seasonality_mode='additive',
    interval_width=0.95,
    yearly_seasonality=True,
    weekly_seasonality=True
)

print("🔮 Training Prophet model...")
prophet_model.train(train_series)
print("✓ Prophet model trained successfully")

# Evaluate on test set
prophet_metrics = prophet_model.evaluate(test_series)
print("\n📊 Prophet Model Metrics:")
for metric, value in prophet_metrics.items():
    print(f"   {metric}: {value:.2f}")

In [ ]:
# Log Prophet to MLflow
with mlflow.start_run(run_name="prophet_baseline"):
    mlflow.log_params({
        "model": "Prophet",
        "seasonality_mode": "additive",
        "train_size": len(train_series),
        "test_size": len(test_series)
    })
    
    for metric, value in prophet_metrics.items():
        mlflow.log_metric(metric, value)
    
    mlflow.log_metric("timestamp", int(datetime.now().timestamp()))

print("✓ Prophet results logged to MLflow")

## 4. Train LSTM Model

In [ ]:
# Initialize and train LSTM
lstm_model = LSTMForecaster(
    units=settings.LSTM_UNITS,
    dropout=settings.LSTM_DROPOUT,
    batch_size=settings.LSTM_BATCH_SIZE,
    epochs=settings.LSTM_EPOCHS,
    lookback=30
)

print("🧠 Training LSTM model...")
lstm_model.train(train_series, verbose=0)
print("✓ LSTM model trained successfully")

# Evaluate on test set
lstm_metrics = lstm_model.evaluate(test_series)
print("\n📊 LSTM Model Metrics:")
for metric, value in lstm_metrics.items():
    print(f"   {metric}: {value:.2f}")

In [ ]:
# Log LSTM to MLflow
with mlflow.start_run(run_name="lstm_baseline"):
    mlflow.log_params({
        "model": "LSTM",
        "units": settings.LSTM_UNITS,
        "dropout": settings.LSTM_DROPOUT,
        "epochs": settings.LSTM_EPOCHS,
        "lookback": 30
    })
    
    for metric, value in lstm_metrics.items():
        mlflow.log_metric(metric, value)

print("✓ LSTM results logged to MLflow")

## 5. Ensemble Model & Weight Optimization

In [ ]:
# Create ensemble
ensemble = EnsembleForecaster(
    prophet_weight=0.5,
    lstm_weight=0.5
)

# Train ensemble
print("🎯 Training ensemble model...")
ensemble.train(train_series, verbose=0)
print("✓ Ensemble model trained successfully")

# Evaluate ensemble
ensemble_metrics = ensemble.evaluate(test_series)
print("\n📊 Ensemble Model Metrics (Initial Weights):")
for model_name, metrics in ensemble_metrics.items():
    print(f"\n{model_name.upper()}:")
    for metric, value in metrics.items():
        print(f"   {metric}: {value:.2f}")

In [ ]:
# Optimize ensemble weights
print("\n⚙️  Optimizing ensemble weights...")
ensemble.optimize_weights(test_series, n_iterations=20)

print(f"\n✓ Optimized weights:")
print(f"   Prophet: {ensemble.prophet_weight:.3f}")
print(f"   LSTM: {ensemble.lstm_weight:.3f}")

In [ ]:
# Log ensemble to MLflow
with mlflow.start_run(run_name="ensemble_optimized"):
    mlflow.log_params({
        "model": "Ensemble",
        "prophet_weight": ensemble.prophet_weight,
        "lstm_weight": ensemble.lstm_weight,
        "optimization_iterations": 20
    })
    
    # Log combined metrics
    for model_name, metrics in ensemble_metrics.items():
        for metric, value in metrics.items():
            mlflow.log_metric(f"{model_name}_{metric}", value)

print("✓ Ensemble results logged to MLflow")

## 6. Model Comparison

In [ ]:
# Create comparison dataframe
comparison_data = {
    'Prophet': prophet_metrics,
    'LSTM': lstm_metrics,
    'Ensemble': ensemble_metrics['prophet']  # Use prophet metrics as reference
}

comparison_df = pd.DataFrame(comparison_data)

print("\n📊 Model Performance Comparison")
print(comparison_df.round(2))

# Visualize comparison
fig = go.Figure()

for model in ['Prophet', 'LSTM', 'Ensemble']:
    fig.add_trace(go.Bar(
        name=model,
        x=list(comparison_df.index),
        y=comparison_df[model].values,
        text=comparison_df[model].round(2).values,
        textposition='auto'
    ))

fig.update_layout(
    title='Model Performance Comparison',
    xaxis_title='Metric',
    yaxis_title='Value',
    barmode='group',
    height=500,
    hovermode='x unified'
)
fig.show()

## 7. Generate Forecasts

In [ ]:
# Generate 7-day ensemble forecast
print("📈 Generating 7-day forecast...")
ensemble_forecast = ensemble.forecast(series, periods=7)

print(f"\n✓ Forecast generated")
print(f"   Forecast dates: {ensemble_forecast['dates'][0]} to {ensemble_forecast['dates'][-1]}")
print(f"\n7-Day Forecast Results:")

forecast_df = pd.DataFrame({
    'Date': ensemble_forecast['dates'],
    'Ensemble': ensemble_forecast['ensemble'],
    'Prophet': ensemble_forecast['prophet'],
    'LSTM': ensemble_forecast['lstm'],
    'Lower_CI': ensemble_forecast['lower'],
    'Upper_CI': ensemble_forecast['upper']
})

print(forecast_df.round(0))

## 8. Visualization - Actual vs Predicted

In [ ]:
# Get predictions on test set
prophet_pred = prophet_model.forecast(len(test_series))
lstm_pred = lstm_model.forecast(train_series, len(test_series))

# Create visualization
fig = go.Figure()

# Add actual values
fig.add_trace(go.Scatter(
    x=test_series.index,
    y=test_series.values,
    name='Actual',
    mode='lines',
    line=dict(color='#000000', width=2)
))

# Add Prophet predictions
fig.add_trace(go.Scatter(
    x=prophet_pred.index,
    y=prophet_pred['yhat'].values,
    name='Prophet',
    mode='lines',
    line=dict(color='#1f77b4', width=2, dash='dash')
))

# Add LSTM predictions
fig.add_trace(go.Scatter(
    x=test_series.index,
    y=lstm_pred,
    name='LSTM',
    mode='lines',
    line=dict(color='#ff7f0e', width=2, dash='dash')
))

fig.update_layout(
    title='Test Set: Actual vs Predicted Values',
    xaxis_title='Date',
    yaxis_title='Transactions',
    height=600,
    hovermode='x unified'
)
fig.show()

# Calculate residuals
prophet_residuals = test_series.values - prophet_pred['yhat'].values
lstm_residuals = test_series.values - lstm_pred

# Residual statistics
print("\n📊 Residual Analysis")
print("\nProphet:")
print(f"   Mean: {np.mean(prophet_residuals):.2f}")
print(f"   Std Dev: {np.std(prophet_residuals):.2f}")
print(f"   Min: {np.min(prophet_residuals):.2f}")
print(f"   Max: {np.max(prophet_residuals):.2f}")

print("\nLSTM:")
print(f"   Mean: {np.mean(lstm_residuals):.2f}")
print(f"   Std Dev: {np.std(lstm_residuals):.2f}")
print(f"   Min: {np.min(lstm_residuals):.2f}")
print(f"   Max: {np.max(lstm_residuals):.2f}")

# Residual plots
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Prophet Residuals', 'LSTM Residuals')
)

fig.add_trace(go.Histogram(x=prophet_residuals, name='Prophet', nbinsx=30,
                           marker_color='#1f77b4'), row=1, col=1)
fig.add_trace(go.Histogram(x=lstm_residuals, name='LSTM', nbinsx=30,
                           marker_color='#ff7f0e'), row=1, col=2)

fig.update_layout(height=400, title_text="Residual Distributions",
                  showlegend=True, hovermode='x unified')
fig.show()

## 9. Summary & Next Steps

print("\n✅ Model Training Complete!\n")
print("📊 Results Summary:")
print(f"  • Prophet MAPE: {prophet_metrics['MAPE']:.2f}%")
print(f"  • LSTM MAPE: {lstm_metrics['MAPE']:.2f}%")
print(f"  • Ensemble Prophet Weight: {ensemble.prophet_weight:.2%}")
print(f"  • Ensemble LSTM Weight: {ensemble.lstm_weight:.2%}")
print(f"\n🎯 Best Performing Model: {'Prophet' if prophet_metrics['MAPE'] < lstm_metrics['MAPE'] else 'LSTM'}")
print(f"\n📈 7-Day Forecast Ready for Production")
print(f"\n🔗 MLflow Experiment: {settings.MLFLOW_TRACKING_URI}")